In [1]:
# Memory safety: cap this kernel to the RAM free right now so an out-of-memory
# feature build fails with a clean MemoryError instead of crashing VS Code /
# thrashing swap. This notebook builds cross-row features (lags/rolling), which
# can't be row-batched, so the guard is the protection here.
import os, sys
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")) ; from memory_compute import *
install_memory_guard()


[memory_guard] hard cap 13.3G virtual on this kernel (total RAM 14.8G, 9.7G free now). Runaway allocations fail cleanly; bounded streaming keeps normal work well under this.


14298148864

In [2]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables

REGION = variables.TARGET_REGION

PER_HOUR = 60 // variables.FEATURE_GRANULARITY_IN_MINUTES   # 12
PER_DAY  = 24 * PER_HOUR                                     # 288

# Offer-PRICE behaviour of the regional bid stack. Using the 10 band prices
# (_prices, $/MWh from BIDDAYOFFER_D) weighted by the MW offered in each band
# (_bands, from BIDPEROFFER_D) we measure how expensively generators are
# offering their capacity, and how they *re-price* it over time (rebidding).
# Bids for interval T are known at/before T, so all features are leakage-free.
PRICE_THRESHOLDS = [300, 1000, 5000]
FUEL_KEYS = {"Coal": "coal", "Gas": "gas", "Hydro": "hydro", "Battery Storage": "battery"}

_mapping = pd.read_parquet("../1_Dataset/Processed_data/0_nem_duid_mapping.parquet")
DUID_REGION = dict(zip(_mapping["DUID"], _mapping["Region"]))
DUID_FUEL   = dict(zip(_mapping["DUID"], _mapping["Fuel Source - Primary"]))


def _parse_bands(series: pd.Series) -> np.ndarray:
    """Parse a column of comma-separated 10-band strings into a (T, 10) array."""
    return series.str.split(",", expand=True).to_numpy(dtype=np.float32)

In [3]:
import pyarrow.parquet as pq

# Bids are stored per DUID: {DUID}_bands (MW) in 8_bid_availability and
# {DUID}_prices ($/MWh) in 8_bid_prices.
BANDS_PATH  = "../1_Dataset/Processed_data/8_bid_availability.parquet"
PRICES_PATH = "../1_Dataset/Processed_data/8_bid_prices.parquet"

# These files are ~572 comma-separated-string columns each; loading them whole
# expands to tens of GB of Python strings and is what crashes the kernel. The
# loop below only ever touches one DUID's columns at a time, so we read column
# names from the schema now and pull each DUID's data on demand.
bands_cols  = pq.ParquetFile(BANDS_PATH).schema_arrow.names
prices_cols = pq.ParquetFile(PRICES_PATH).schema_arrow.names

REGION_DUIDS = sorted(
    {c[:-len("_bands")]  for c in bands_cols  if c.endswith("_bands")}
    & {c[:-len("_prices")] for c in prices_cols if c.endswith("_prices")}
)
REGION_DUIDS = [d for d in REGION_DUIDS if DUID_REGION.get(d) == REGION.upper()]

# Use the complete dispatch-price timeline as the 5-minute spine. Sparse bid
# months are reindexed and causally forward-filled per DUID below.
bands_index = pd.read_parquet(
    "../1_Dataset/Processed_data/1_dispatch_price.parquet", columns=[]
).index

df = pd.DataFrame(index=bands_index)
df_core_columns = df.columns
print(f"{REGION}: {len(REGION_DUIDS)} DUIDs")
df[:10]

nsw: 110 DUIDs


""
Date
2018-01-01 00:05:00
2018-01-01 00:10:00
2018-01-01 00:15:00
2018-01-01 00:20:00
2018-01-01 00:25:00
2018-01-01 00:30:00
2018-01-01 00:35:00
2018-01-01 00:40:00
2018-01-01 00:45:00


In [4]:
def _add_offer_price_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    How expensively the region is offering its capacity, right now. The
    capacity-weighted average offer price is the MW-weighted mean band price
    across all in-region DUIDs; the marginal offer price is the highest band
    price still carrying MW (the ceiling of the offered stack). We also measure
    how much MW is offered above $300 / $1000 / $5000, plus a per-fuel
    capacity-weighted price (coal / gas / hydro / battery). Leakage-free.
    Returns only the new columns to avoid copying the full base frame.
    """
    n_rows = len(df.index)
    num = np.zeros(n_rows)
    den = np.zeros(n_rows)
    marginal = np.full(n_rows, -np.inf)
    above = {th: np.zeros(n_rows) for th in PRICE_THRESHOLDS}
    fuel_num = {k: np.zeros(n_rows) for k in set(FUEL_KEYS.values())}
    fuel_den = {k: np.zeros(n_rows) for k in set(FUEL_KEYS.values())}

    for duid in REGION_DUIDS:
        b_series = pd.read_parquet(BANDS_PATH, columns=[f"{duid}_bands"])[f"{duid}_bands"].reindex(df.index).ffill()
        b = _parse_bands(b_series)
        p_series = pd.read_parquet(PRICES_PATH, columns=[f"{duid}_prices"])[f"{duid}_prices"].reindex(df.index).ffill()
        p = _parse_bands(p_series)
        m = min(len(b), len(p))
        b, p = b[:m], p[:m]

        num[:m] += np.nansum(b * p, axis=1)
        den[:m] += np.nansum(b, axis=1)
        for th in PRICE_THRESHOLDS:
            above[th][:m] += np.nansum(np.where(p > th, b, 0.0), axis=1)
        offered_price = np.where(b > 0, p, -np.inf).max(axis=1)
        marginal[:m] = np.maximum(marginal[:m], offered_price)

        fuel_key = FUEL_KEYS.get(DUID_FUEL.get(duid))
        if fuel_key:
            fuel_num[fuel_key][:m] += np.nansum(b * p, axis=1)
            fuel_den[fuel_key][:m] += np.nansum(b, axis=1)

    new_cols = {}
    new_cols[f"bidprice_{REGION}_capwtd"]   = np.where(den > 0, num / den, np.nan).astype(np.float32)
    new_cols[f"bidprice_{REGION}_marginal"] = np.where(np.isfinite(marginal), marginal, np.nan).astype(np.float32)
    for th in PRICE_THRESHOLDS:
        new_cols[f"bidprice_{REGION}_mw_above_{th}"] = above[th].astype(np.float32)
    for k in fuel_num:
        new_cols[f"bidprice_{REGION}_{k}_capwtd"] = np.where(fuel_den[k] > 0, fuel_num[k] / fuel_den[k], np.nan).astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)


new_df = _add_offer_price_features(df)
df = pd.concat([df, new_df], axis=1)
new_df[:10]


/tmp/ipykernel_51945/449428975.py:40: RuntimeWarning: invalid value encountered in divide
  new_cols[f"bidprice_{REGION}_capwtd"]   = np.where(den > 0, num / den, np.nan).astype(np.float32)
/tmp/ipykernel_51945/449428975.py:45: RuntimeWarning: invalid value encountered in divide
  new_cols[f"bidprice_{REGION}_{k}_capwtd"] = np.where(fuel_den[k] > 0, fuel_num[k] / fuel_den[k], np.nan).astype(np.float32)


,bidprice_nsw_capwtd,bidprice_nsw_marginal,bidprice_nsw_mw_above_300,bidprice_nsw_mw_above_1000,bidprice_nsw_mw_above_5000,bidprice_nsw_hydro_capwtd,bidprice_nsw_gas_capwtd,bidprice_nsw_coal_capwtd,bidprice_nsw_battery_capwtd
Date,,,,,,,,,
2018-01-01 00:05:00,3930.698975,14703.0,6273.0,4713.0,4713.0,8506.432617,10867.061523,390.915894,NaN
2018-01-01 00:10:00,3930.698975,14703.0,6273.0,4713.0,4713.0,8506.432617,10867.061523,390.915894,NaN
2018-01-01 00:15:00,3930.698975,14703.0,6273.0,4713.0,4713.0,8506.432617,10867.061523,390.915894,NaN
2018-01-01 00:20:00,3930.698975,14703.0,6273.0,4713.0,4713.0,8506.432617,10867.061523,390.915894,NaN
2018-01-01 00:25:00,3930.698975,14703.0,6273.0,4713.0,4713.0,8506.432617,10867.061523,390.915894,NaN
2018-01-01 00:30:00,3930.698975,14703.0,6273.0,4713.0,4713.0,8506.432617,10867.061523,390.915894,NaN
2018-01-01 00:35:00,4088.245850,14703.0,6193.0,4833.0,4833.0,9297.243164,10867.884766,391.082947,NaN
2018-01-01 00:40:00,4088.245850,14703.0,6193.0,4833.0,4833.0,9297.243164,10867.884766,391.082947,NaN
2018-01-01 00:45:00,4088.245850,14703.0,6193.0,4833.0,4833.0,9297.243164,10867.884766,391.082947,NaN


In [5]:
def _add_price_rebidding_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Price rebidding dynamics: how the capacity-weighted offer price and the
    high-price MW share move versus the previous interval, the last hour and the
    same time yesterday. A sudden lift in the weighted offer price or in MW
    re-offered above $300 is a leading signal of a tightening / spiking market.
    A short rolling mean captures the prevailing offer-price regime. All
    look-backs are backward-only, hence leakage-free.
    Returns only the new columns to avoid copying the full base frame.
    """
    capwtd    = df[f"bidprice_{REGION}_capwtd"]
    above_300 = df[f"bidprice_{REGION}_mw_above_300"]

    new_cols = {}
    for lag, lab in [(1, "5m"), (PER_HOUR, "1h"), (PER_DAY, "1d")]:
        new_cols[f"bidprice_{REGION}_capwtd_rebid_{lab}"]  = capwtd.diff(lag).astype(np.float32)
        new_cols[f"bidprice_{REGION}_above300_rebid_{lab}"] = above_300.diff(lag).astype(np.float32)

    new_cols[f"bidprice_{REGION}_capwtd_rmean_1d"] = capwtd.rolling(PER_DAY, min_periods=1).mean().astype(np.float32)
    new_cols[f"bidprice_{REGION}_capwtd_pctrank_1d"] = (
        capwtd.rolling(PER_DAY, min_periods=PER_HOUR).rank(pct=True).astype(np.float32)
    )
    return pd.DataFrame(new_cols, index=df.index)


new_df = _add_price_rebidding_features(df)
df = pd.concat([df, new_df], axis=1)
new_df[:10]


,bidprice_nsw_capwtd_rebid_5m,bidprice_nsw_above300_rebid_5m,bidprice_nsw_capwtd_rebid_1h,bidprice_nsw_above300_rebid_1h,bidprice_nsw_capwtd_rebid_1d,bidprice_nsw_above300_rebid_1d,bidprice_nsw_capwtd_rmean_1d,bidprice_nsw_capwtd_pctrank_1d
Date,,,,,,,,
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,3930.698975,NaN
2018-01-01 00:10:00,0.000000,0.0,NaN,NaN,NaN,NaN,3930.698975,NaN
2018-01-01 00:15:00,0.000000,0.0,NaN,NaN,NaN,NaN,3930.698975,NaN
2018-01-01 00:20:00,0.000000,0.0,NaN,NaN,NaN,NaN,3930.698975,NaN
2018-01-01 00:25:00,0.000000,0.0,NaN,NaN,NaN,NaN,3930.698975,NaN
2018-01-01 00:30:00,0.000000,0.0,NaN,NaN,NaN,NaN,3930.698975,NaN
2018-01-01 00:35:00,157.546875,-80.0,NaN,NaN,NaN,NaN,3953.205566,NaN
2018-01-01 00:40:00,0.000000,0.0,NaN,NaN,NaN,NaN,3970.085693,NaN
2018-01-01 00:45:00,0.000000,0.0,NaN,NaN,NaN,NaN,3983.214600,NaN


In [6]:
print("Total features:", df.shape[1])
df = df.drop(columns=df_core_columns)
df.to_parquet("../2_Features_build/Feature_data/8_2_bid_prices.parquet")
df.shape

Total features: 17


(893664, 17)

In [7]:
# Free this kernel's memory so the next notebook has RAM to work with
# (clears data variables + returns freed heap to the OS).
release_memory()


[release_memory] cleared 21 variable(s); kernel rss 0.59G, 9.1G RAM free now
